## StatsBomb Open Data — Germany (Nagelsmann era, Euro 2024)

**Julian Nagelsmann** became Germany men’s head coach in **September 2023**. This notebook uses **free [StatsBomb Open Data](https://github.com/statsbomb/open-data)** only (JSON on GitHub — no Hudl API key).

**What Open Data includes for this story:** **UEFA Euro 2024** (`competition_id` 55, `season_id` 282) fits his Germany tenure (tournament June–July 2024).

**What it does *not* include:** Nagelsmann’s **Bayern** seasons in **1. Bundesliga** are **not** in the free dataset in the same way (Open Data only carries selected Bundesliga seasons). We do **not** mix paid or unlisted competitions here.

In [1]:
import pandas as pd
import requests

OPEN_DATA = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"

EURO_2024_COMPETITION_ID = 55
EURO_2024_SEASON_ID = 282
GERMANY = "Germany"

### Load competitions (sanity check)
Confirm the Euro 2024 row exists before downloading matches.

In [2]:
comp_url = f"{OPEN_DATA}/competitions.json"
cr = requests.get(comp_url)
cr.raise_for_status()
comp = pd.DataFrame(cr.json())
row = comp[
    (comp["competition_id"] == EURO_2024_COMPETITION_ID)
    & (comp["season_id"] == EURO_2024_SEASON_ID)
]
row[["competition_name", "season_name", "competition_id", "season_id"]]

,competition_name,season_name,competition_id,season_id
68,UEFA Euro,2024,55,282


### Germany matches at Euro 2024
Filter the match list where **Germany** is home or away.

In [3]:
match_url = f"{OPEN_DATA}/matches/{EURO_2024_COMPETITION_ID}/{EURO_2024_SEASON_ID}.json"
r = requests.get(match_url)
r.raise_for_status()
matches = pd.json_normalize(r.json())

germany_mask = (
    (matches["home_team.home_team_name"] == GERMANY)
    | (matches["away_team.away_team_name"] == GERMANY)
)
ger = matches.loc[germany_mask].copy()

def opponent(row):
    if row["home_team.home_team_name"] == GERMANY:
        return row["away_team.away_team_name"]
    return row["home_team.home_team_name"]

ger["opponent"] = ger.apply(opponent, axis=1)
out_cols = [
    "match_id",
    "match_date",
    "home_team.home_team_name",
    "away_team.away_team_name",
    "opponent",
    "competition_stage.name",
]
ger = ger[out_cols].sort_values("match_date")
ger

,match_id,match_date,home_team.home_team_name,away_team.away_team_name,opponent,competition_stage.name
50,3930158,2024-06-14,Germany,Scotland,Scotland,Group Stage
25,3930168,2024-06-19,Germany,Hungary,Hungary,Group Stage
40,3930176,2024-06-23,Switzerland,Germany,Switzerland,Group Stage
19,3940983,2024-06-29,Germany,Denmark,Denmark,Round of 16
8,3942226,2024-07-05,Spain,Germany,Spain,Quarter-finals


In [4]:
from pathlib import Path

ROOT = Path.cwd()
ids_path = ROOT / "statsbomb" / "germany_euro2024_match_ids.csv"
if not ids_path.parent.is_dir():
    ids_path = ROOT / "germany_euro2024_match_ids.csv"
ger[["match_id"]].to_csv(ids_path, index=False)